# Programmieraufgabe 5

In dieser Aufgabe wollen wir - wie in Aufgabe 4 - die Bahn einer Rakete zwischen Erde und Mond berechnen. Dabei soll diesmal Schrittweitensteuerung eingesetzt werden.

Tragen Sie zunächst in der folgenen Zelle Ihren Namen ein:

In [ ]:
# Numerik gewöhnlicher Differentialgleichungen
# Sommersemester 2026
# Übungsblatt 8 - Programmieraufgabe 5
#
# [Nachname], [Vorname]
# [Vorname.Nachname@uni-a.de]

In [ ]:
import numpy as np
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from IPython.display import HTML

from util.plotting_05 import plot_RK_solution, init_animation, draw_frame
%matplotlib inline

Die folgende Funktion implementiert bereits die Butcher-Tabelle des Dormand-Prince-Verfahrens. Beachten Sie, dass, anders als in der letzten Programmieraufgabe, `b` jetzt aus zwei Zeilen besteht.

In [ ]:
def RK_DP54():
    '''
        Rückgabewert:
            A, b, c: Butcher-Tabelle für das Dormand-Prince-Verfahren mit Ordnung 5 und 4
    '''
    A = np.zeros((7,7))
    A[1,0] = 0.2
    A[2,0] = 0.075
    A[2,1] = 0.225
    A[3,0] = 44/45
    A[3,1] = -56/15
    A[3,2] = 32/9
    A[4,0] = 19372/6561
    A[4,1] = -25360/2187
    A[4,2] = 64448/6561
    A[4,3] = -212/729
    A[5,0] = 9017/3168
    A[5,1] = -355/33
    A[5,2] = 46732/5247
    A[5,3] = 49/176
    A[5,4] = -5103/18656
    A[6,0] = 35/384
    A[6,1] = 0.0
    A[6,2] = 500/1113
    A[6,3] = 125/192
    A[6,4] = -2187/6784
    A[6,5] = 11/84

    b = np.array ([
        [5179/57600, 0.0, 7571/16695, 393/640, -92097/339200, 187/2100, 0.025], # Ordnung 4
        [    35/384, 0.0,   500/1113, 125/192,    -2187/6784,    11/84,   0.0], # Ordnung 5
    ]) 

    c = np.array([0.0, 0.2, 0.3, 0.8, 8/9, 1.0, 1.0])
    return A,b,c 

Vervollständigen Sie jetzt die folgende Implementierung des Runge-Kutta-Verfahrens mit Schrittweitensteuerung (Algorithmus 4.8). Beachten Sie dabei:
- Anders als in der letzten Aufgabe soll die Funktion jetzt nicht nur den nächsten Schritt, sondern alle Schritte bis zum Endzeitpunkt `t_final` zurückgeben.
- Dazu sollen zwei Arrays zurückgegeben werden: `us`, das die Folge der berechneten Werte enthält, und `hs`, mit den dazugehörigen Schrittweiten. Dabei ist `us[i] = us[i-1] + h[i]`. Für `us[0] = u0` gibt es also keine Schrittweite, deswegen setzen wir `hs[0] = np.nan`.
- Um Endlosschleifen zu vermeiden, werden maximal `max_not_accepted` viele Korrekturen pro Zeitschritt durchgeführt. Danach wird der aktuelle Schrittweitenvorschlag angenommen, selbst wenn `delta` größer ist als die vorgegebene Toleranz.
- Da bei eingebetteten Verfahren nicht im Vorhinein klar ist, wie viele Schritte am Ende benötigt werden, müssen die Ausgabe-Arrays evtl. verlängert werden. Da das aber teuer ist, verlängern wir sie um `blocksize` viele Einträge auf einmal und kürzen die Arrays am Ende auf die tatsächliche Größe. Das ist bereits für Sie implementiert.

In [ ]:
def solve_RKF(f, t0, t_final, u0, h0, Butcher, p, step_params=(1e-4, 0.9, 5.0, 0.1, 50), blocksize=1000):
    '''
        Berechnet mit dem Runge-Kutta-Verfahren mit Schrittweitensteuerung
        mit gegebener Butcher Tabelle die Lösung der
        Differentialgleichung y'(t) = f(t, y(t))
        Parameter:
            f           : rechte Seite der Differentialgleichung;
            t0          : Start-Zeitpunkt
            t_final     : End-Zeitpunkt
            u0          : Start-Funktionswert
            h0          : Start-Schrittweite
            Butcher     : Butcher-Schema A, b, c, mit zwei Zeilen für b
            p           : Ordnung des 'schlechteren' Teil-Verfahrens
            step_params : Parameter der Schrittweitensteuerung:
                eps              : vorgegebene Toleranz für den Fehler-Schätzer
                rho              : Schrittweitenverkleinerungsfaktor
                q                : Schrittweitenvergrößerungsfaktor
                h_max            : maximale Schrittweite
                max_not_accepted : Maximale Anzahl von Schrittweitenkorrekturen pro Zeitschritt
            blocksize   : Anzahl der Einträge, um die die Arrays verlängert werden
        Ausgabe-Parameter:
            us        : berechneter Loesungspfad
            hs        : Schrittweiten
    '''
    A, b, c = Butcher
    eps, rho, q, h_max, max_not_accepted = step_params
    
    n  = u0.shape[0]
    m  =  c.shape[0]
    
    k  = np.zeros((???))
    hs = np.zeros(blocksize)
    us = np.zeros((blocksize, n))
    hs[0]   = np.nan # wir speichern keine Schrittweite für den Anfangswert
    us[0,:] = u0
    t_cur   = t0
    u_cur   = u0.copy()
    h_cur   = h0

    i = 0
    n_not_accepted = 0
    while ???:
        h_cur = ???

        # Berechne k
        ???

        # Berechne den nächsten Schritt in beiden Verfahren
        u_new_1 = u_cur + ???
        u_new_2 = u_cur + ???

        e     = min(1e-16, eps)    # um Teilen durch 0 zu vermeiden
        delta = max(e, np.linalg.norm(u_new_1 - u_new_2))

        # Bestimme die neue Schrittweite
        h_new = ???

        accepted = ???
        if not accepted:
            n_not_accepted += 1
            if n_not_accepted > max_not_accepted:
                ???
            else:       # neuer Versuch mit neuer Schrittweite
                h_cur = ???
        if accepted:
            n_not_accepted = 0            
            i = i + 1
            if i % blocksize == 0:
                hs = np.append(hs, np.zeros(blocksize))
                us = np.append(us, np.zeros((blocksize, n)), axis=0)
            t_cur  = ???
            u_cur  = ???
            us[i,:] = ???
            hs[i]  = ???
            h_cur  = ???

    return us[:i+1,:], hs[:i+1]

In [ ]:
# Wir könnten zur Berechnung von `y_new_1` und `y_new_2` auch direkt die Funktion RK()
# aus Programmieraufgabe 4 verwenden. Warum wäre das aber deutlich ineffizienter?
#
#
#

Das Modell ist wieder das selbe wie in der vorherigen Aufgabe.

In [ ]:
def Erde_Mond_Rakete(t, y, mu):
    '''
        Rechte Seite der Differentialgleichung, die die Bahn 
        einer Raumsonde zwischen Erde und Mond beschreibt
        Parameter:
            t     : aktueller Zeitpunkt
            y     : aktueller Funktionswert (x_1, x'_1, x_2, x'_2)
            mu    : relative Masse des Mondes im Vergleich zur Masse Erde+Mond
        Rückgabewert:
            y_new : Wert der Funktion
    '''
    
    mu_hat = 1.0 - mu # relative Masse der Erde
    n1 =  ((y[0] + mu    )**2 + y[2] * y[2])**1.5
    n2 =  ((y[0] - mu_hat)**2 + y[2] * y[2])**1.5

    y_new = np.array([y[1], y[0], y[3], y[2]])
    y_new[1] += 2 * y[3] - mu_hat * (y[0] + mu) / n1 - mu * (y[0] - mu_hat) / n2
    y_new[3] -= 2 * y[1] + mu_hat *  y[2]       / n1 + mu *  y[2]           / n2
    return y_new

Berechnen Sie jetzt mit Ihrem Löser mit Schrittweitensteuerung die Bahn der Raumsonde, mit einer Start-Schrittweite von `1e-4` und einer maximalen Schrittweite `0.1`. Ergänzen Sie dazu in der nächsten Zelle die notwendigen Werte.

In [ ]:
mu = 0.012277470841006752  # relative Masse des Mondes
f = lambda t, u: Erde_Mond_Rakete(t, u, mu)

# Anfangswert
y0  = np.array([0.994, 0.0, 0.0, -2.001585106])
t0  = 0
T   = 17.0652166

p   = ???
h0  = ???
step_params = (1e-10, 0.9, 5.0, ???, 5)

ys, step_sizes = solve_RKF(???, RK_DP54(), p, step_params=step_params)

fig = plt.figure(figsize=(5, 8))
ts = t0 + np.array([0, *np.cumsum(step_sizes[1:])])
plot_RK_solution(ys, ts, step_sizes, mu, xlims = (-1.5, 1.2), ylims=(-1.5, 1.5))

Abschließend können wir auch wieder eine Animation erzeugen. Das Plotten kann dabei einige Momente dauern.

In [ ]:
n_steps = ys.shape[0] - 1
steps_per_frame = 3
frames = n_steps // steps_per_frame
u = ys[:,[0,2]]
v = ys[:,[1,3]]

plt.ioff()
fig, ax = plt.subplots(2)
fig.set_figwidth(6)
fig.set_figheight(8)
plt.title(f'Orbit  mit Dormand-Prince, {n_steps} Schritte')
init_func = lambda: init_animation(ax, mu, u[0,:], v[0,:], ts, step_sizes)
func = lambda frame: draw_frame(frame, ax, steps_per_frame, u, 0.2*v, ts, step_sizes)    # v wird für das Plotten skaliert
anim = animation.FuncAnimation(fig=fig, func=func, frames=frames, init_func=init_func, interval=50)
plt.ion()

HTML(anim.to_jshtml())

In [ ]:
# Warum wird die Animation an manchen Stellen langsamer? Warum genau dort?
#
#
#